<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/CreateAgent/ipynb/1.4_multimodal_messages.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Text input

In [ ]:
%%bash
apt install -y zstd pciutils lshw
curl -fsSL https://ollama.com/install.sh | sh
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core \
    langchain-community langchain-ollama ollama \
    tavily-python

In [ ]:
!nohup ollama serve &

In [ ]:
!ollma pull gpt-oss:20b

In [1]:
import warnings
warnings.filterwarnings("ignore")
import dotenv

_ =  dotenv.load_dotenv(dotenv_path=".env", override=True)

In [7]:
import langchain_ollama
from langchain import agents, messages
import time


model = langchain_ollama.ChatOllama(
    model="gpt-oss:120b-cloud",
    base_url="https://ollama.com",
    headers={"Authorization": f"Bearer {os.environ.get('OLLAMA_API_KEY')}"},
    temperature=1.0,
    max_tokens=128,
)

agent = agents.create_agent(model=model)

question = messages.HumanMessage(content="""
    What's the capital of the moon?
""")

start_time = time.time()
response = agent.invoke(input={"messages": [question]})
print(response['messages'][-1].content)
end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

The Moon isn’t a country, so it doesn’t have an official capital the way a nation does. There’s no government‑designated “capital city” on our nearest celestial neighbor.

That said, the idea of a lunar capital pops up a lot in science‑fiction and in real‑world planning:

| Context | “Capital” that’s been mentioned |
|---------|---------------------------------|
| **Science‑fiction** | • **Luna City** – a sprawling settlement in Robert A. Heinlein’s *The Moon Is a Harsh Mistress* (often called the “capital of the Lunar Republic”). <br>• **Tycho Base** – the political hub of the Belt in *The Expanse* series. |
| **Pop‑culture / games** | • **Selene Base**, **Artemis Base**, **Tranquility Base** (the site where Apollo 11 landed) are sometimes portrayed as administrative centers. |
| **Real‑world proposals** | • **Artemis Base Camp** (planned NASA/ESA outpost near the lunar south pole) is being discussed as the “hub” for future lunar operations. <br>• The International Lunar Research Stat

In [ ]:
import langchain_ollama
from langchain import agents, messages

model = langchain_ollama.ChatOllama(
    model="gpt-oss
)

agent = agents.create_agent(
    model='gpt-5-nano',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

## Image input

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

In [ ]:
print(uploader.value)

In [ ]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

## Audio input

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

In [ ]:
agent = create_agent(
    model='gpt-4o-audio-preview',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)